In [1]:
from scapy.all import rdpcap, PcapReader

# 方式1: 一次性读取所有数据包(适合小文件)
packets = rdpcap('apollo_eth0_sample_0.pcap')
print(f"总共有 {len(packets)} 个数据包")

总共有 563 个数据包


In [4]:
print(packets[0].show())

###[ Ethernet ]###
  dst       = 33:33:ff:20:d3:83
  src       = ba:39:47:20:d3:83
  type      = IPv6
###[ IPv6 ]###
     version   = 6
     tc        = 0
     fl        = 0
     plen      = 32
     nh        = ICMPv6
     hlim      = 255
     src       = ::
     dst       = ff02::1:ff20:d383
###[ ICMPv6 Neighbor Discovery - Neighbor Solicitation ]###
        type      = Neighbor Solicitation
        code      = 0
        cksum     = 0x5de8
        res       = 0
        tgt       = fe80::b839:47ff:fe20:d383
###[ ICMPv6 Neighbor Discovery Option - Scapy Unimplemented ]###
           type      = 14
           len       = 1
           data      = b'a\x01\x9d:lx'

None


In [5]:
from scapy.all import rdpcap, Packet
import json

def packet_to_dict(pkt):
    """递归提取数据包的所有层和字段"""
    packet_dict = {
        'summary': pkt.summary(),
        'timestamp': float(pkt.time) if hasattr(pkt, 'time') else None,
        'length': len(pkt),
        'layers': []
    }
    
    # 遍历所有协议层
    layer = pkt
    while layer:
        layer_dict = {
            'layer_name': layer.name,
            'fields': {}
        }
        
        # 提取该层的所有字段
        for field_name, field_value in layer.fields.items():
            # 处理不同类型的字段值
            if isinstance(field_value, bytes):
                layer_dict['fields'][field_name] = field_value.hex()
            elif isinstance(field_value, Packet):
                # 如果字段值是另一个数据包，递归处理
                layer_dict['fields'][field_name] = packet_to_dict(field_value)
            else:
                layer_dict['fields'][field_name] = str(field_value)
        
        packet_dict['layers'].append(layer_dict)
        layer = layer.payload if layer.payload else None
    
    return packet_dict

# 读取并转换
packets = rdpcap('apollo_eth0_sample_0.pcap')
packets_json = []

for i, pkt in enumerate(packets):
    packet_data = packet_to_dict(pkt)
    packet_data['packet_index'] = i
    packets_json.append(packet_data)

# 保存为 JSON 文件
with open('packets.json', 'w', encoding='utf-8') as f:
    json.dump(packets_json, f, indent=2, ensure_ascii=False)

print(f"成功转换 {len(packets_json)} 个数据包")

成功转换 563 个数据包


In [ ]:
from scapy.all import rdpcap, Packet
import json
import math

def packet_to_dict(pkt):
    """递归提取数据包的所有层和字段"""
    packet_dict = {
        'summary': pkt.summary(),
        'timestamp': float(pkt.time) if hasattr(pkt, 'time') else None,
        'length': len(pkt),
        'layers': []
    }
    
    # 遍历所有协议层
    layer = pkt
    while layer:
        layer_dict = {
            'layer_name': layer.name,
            'fields': {}
        }
        
        # 提取该层的所有字段
        for field_name, field_value in layer.fields.items():
            # 处理不同类型的字段值
            if isinstance(field_value, bytes):
                layer_dict['fields'][field_name] = field_value.hex()
            elif isinstance(field_value, Packet):
                # 如果字段值是另一个数据包，递归处理
                layer_dict['fields'][field_name] = packet_to_dict(field_value)
            else:
                layer_dict['fields'][field_name] = str(field_value)
        
        packet_dict['layers'].append(layer_dict)
        layer = layer.payload if layer.payload else None
    
    return packet_dict

# 读取数据包
packets = rdpcap('apollo_eth0_sample_0.pcap')
total_packets = len(packets)
num_files = 10

# 计算每个文件应包含的数据包数量
packets_per_file = math.ceil(total_packets / num_files)

print(f"总共 {total_packets} 个数据包，将拆分为 {num_files} 个文件")
print(f"每个文件约 {packets_per_file} 个数据包")

filename_list=[]
# 拆分并保存
for file_index in range(num_files):
    start_idx = file_index * packets_per_file
    end_idx = min(start_idx + packets_per_file, total_packets)
    
    # 如果起始索引已经超出范围，跳出循环
    if start_idx >= total_packets:
        break
    
    packets_json = []
    for i in range(start_idx, end_idx):
        packet_data = packet_to_dict(packets[i])
        packet_data['packet_index'] = i  # 保持原始索引
        packets_json.append(packet_data)
    
    # 保存为单独的 JSON 文件
    filename = f'packets_part_{file_index + 1:02d}.json'
    filename_list.append(filename)
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(packets_json, f, indent=2, ensure_ascii=False)
    
    print(f"✓ {filename}: 包含数据包 {start_idx}-{end_idx-1} ({len(packets_json)} 个)")

print(f"\n成功拆分完成！")

总共 563 个数据包，将拆分为 10 个文件
每个文件约 57 个数据包
✓ packets_part_01.json: 包含数据包 0-56 (57 个)
✓ packets_part_02.json: 包含数据包 57-113 (57 个)
✓ packets_part_03.json: 包含数据包 114-170 (57 个)
✓ packets_part_04.json: 包含数据包 171-227 (57 个)
✓ packets_part_05.json: 包含数据包 228-284 (57 个)
✓ packets_part_06.json: 包含数据包 285-341 (57 个)
✓ packets_part_07.json: 包含数据包 342-398 (57 个)
✓ packets_part_08.json: 包含数据包 399-455 (57 个)
✓ packets_part_09.json: 包含数据包 456-512 (57 个)
✓ packets_part_10.json: 包含数据包 513-562 (50 个)

成功拆分完成！


In [ ]:
# 初始化 OpenAI 客户端
client = OpenAI()  # 确保设置了 OPENAI_API_KEY 环境变量

uploaded_file_ids = []

for filename in output_files:
    try:
        print(f"正在上传 {filename}...")
        with open(filename, 'rb') as f:
            file_response = client.files.create(
                file=f,
                purpose='assistants'  # 用于 assistants/vector store
            )
        
        uploaded_file_ids.append(file_response.id)
        print(f"✓ {filename} 上传成功! File ID: {file_response.id}")
        
    except Exception as e:
        print(f"✗ {filename} 上传失败: {str(e)}")

print(f"\n成功上传 {len(uploaded_file_ids)} 个文件！\n")

# ========== 步骤3: 创建或使用现有 Vector Store ==========
print("=" * 60)
print("步骤3: 创建 Vector Store")
print("=" * 60)

# 选项A: 创建新的 Vector Store
try:
    vector_store = client.vector_stores.create(
        name="PCAP Packets Analysis",
        file_ids=[]  # 先创建空的，稍后用 batch 添加文件
    )
    vector_store_id = vector_store.id
    print(f"✓ Vector Store 创建成功! ID: {vector_store_id}\n")
    
except Exception as e:
    print(f"✗ Vector Store 创建失败: {str(e)}\n")
    exit(1)

# 选项B: 如果你已有 Vector Store，可以直接使用
# vector_store_id = "vs_abc123"  # 替换为你的 Vector Store ID
# print(f"使用现有 Vector Store: {vector_store_id}\n")

# ========== 步骤4: 创建 File Batch ==========
print("=" * 60)
print("步骤4: 创建 File Batch")
print("=" * 60)

try:
    # 创建 file batch
    file_batch = client.vector_stores.file_batches.create(
        vector_store_id=vector_store_id,
        file_ids=uploaded_file_ids
    )
    
    print(f"✓ File Batch 创建成功! ID: {file_batch.id}")
    print(f"状态: {file_batch.status}")
    print(f"文件统计: {file_batch.file_counts}\n")
    
    # 等待 batch 处理完成
    print("等待文件处理完成...")
    while file_batch.status in ['in_progress', 'cancelling']:
        time.sleep(5)
        file_batch = client.vector_stores.file_batches.retrieve(
            vector_store_id=vector_store_id,
            batch_id=file_batch.id
        )
        print(f"当前状态: {file_batch.status}, 文件统计: {file_batch.file_counts}")
    
    print(f"\n✓ Batch 处理完成! 最终状态: {file_batch.status}")
    
except Exception as e:
    print(f"✗ File Batch 创建失败: {str(e)}")
    exit(1)